<a href="https://colab.research.google.com/github/hhuang45/OmicSelector/blob/master/AIS_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ruptures

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.2 MB/s eta 0:00:00


In [ ]:
# -*- coding: utf-8 -*-
"""
STA5206: Advanced Clustering Demonstration with TIP and CBTR on AIS Data
Makder: Hssin-Hsiung Huang, Ph.D.
University of Central Florida

DEFINITIVE VERSION: This is the final, comprehensive version of the script. It
includes the Gaussian Mixture Model (GMM) as the Python equivalent of MCLUST,
completing the full suite of requested clustering methods.

Key Features:
1.  **All Methods Included:** Compares CBTR, TIP, GMM (MCLUST), DBSCAN, k-means,
    and a full suite of Hierarchical Clustering methods.
2.  **Comprehensive Reporting:** Provides overall ARI scores and a detailed
    per-vessel-type accuracy breakdown for all algorithms.
3.  **Adaptive CBTR & Realistic Simulation:** Retains the most advanced, physically
    realistic simulation and the fully adaptive CBTR algorithm.
"""

# --- 1. Setup and Library Imports ---
!pip install ruptures -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from IPython.display import display
from collections import defaultdict

# Clustering and evaluation tools
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
from scipy.stats import multivariate_t, poisson
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import adjusted_rand_score

# Change point detection library
import ruptures as rpt

# Set a random seed for reproducibility
np.random.seed(42)


# --- 2. Table Invitation Prior (TIP) Implementation (Tuned) ---
class TIPClustering:
    def __init__(self, n_burn=200, n_samples=200):
        self.n_burn = n_burn; self.n_samples = n_samples; self.total_iter = n_burn + n_samples
        self.posterior_samples = []; self.final_labels = None; self.psm = None; self.k_est = 0
    def _calculate_similarity(self, X):
        dist_matrix = squareform(pdist(X, 'euclidean')); median_dist = np.median(dist_matrix[np.triu_indices(self.n_subjects, k=1)])
        if median_dist == 0: median_dist = 1.0
        self.similarity_matrix = np.exp(-1.0 / median_dist * dist_matrix); self.dist_matrix = dist_matrix
    def _estimate_n_tau(self, r):
        dr = np.sort(np.delete(self.dist_matrix[r, :], r)); algo = rpt.Binseg(model="l2").fit(dr)
        try: chp_locs = algo.predict(n_bkps=2); first_chp = chp_locs[0]
        except rpt.exceptions.NotEnoughPoints: first_chp = 1
        return min(poisson.rvs(mu=max(1, first_chp)), self.n_subjects - 1)
    def _log_posterior_predictive(self, x, cluster_data, p, mu0, lambda0, Psi0, nu0):
        n_k = len(cluster_data); x_bar = np.mean(cluster_data, axis=0) if n_k > 0 else np.zeros(p)
        lambda1 = lambda0 + n_k; nu1 = nu0 + n_k; mu1 = (lambda0 * mu0 + n_k * x_bar) / lambda1
        S_k = np.cov(cluster_data.T) * (n_k - 1) if n_k > 1 else np.zeros((p,p))
        diff_mu0 = (x_bar - mu0).reshape(p, 1); Psi1 = Psi0 + S_k + (lambda0 * n_k / lambda1) * (diff_mu0 @ diff_mu0.T)
        df = nu1 - p + 1; loc = mu1; shape = Psi1 * (lambda1 + 1) / (lambda1 * df)
        try: return multivariate_t.logpdf(x, loc=loc, shape=shape, df=df)
        except np.linalg.LinAlgError: return multivariate_t.logpdf(x, loc=loc, shape=shape + np.eye(p) * 1e-6, df=df)
    def fit(self, X):
        self.n_subjects, self.p = X.shape; self._calculate_similarity(X)
        mu0, lambda0, nu0 = np.mean(X, axis=0), 1.0, self.p
        try: Psi0 = np.linalg.inv(np.cov(X.T) * (self.p - 1))
        except np.linalg.LinAlgError: Psi0 = np.linalg.inv(np.cov(X.T) * (self.p - 1) + np.eye(self.p) * 1e-6)
        c = np.ones(self.n_subjects, dtype=int); print(f"Starting TIP Gibbs Sampler ({self.n_burn} burn-in, {self.n_samples} samples)...")
        for t in range(self.total_iter):
            if (t+1) % 100 == 0: print(f"Iteration {t+1}/{self.total_iter}")
            r_idx = np.random.randint(0, self.n_subjects); n_tau_r = self._estimate_n_tau(r_idx)
            similar_indices = np.argsort(self.similarity_matrix[r_idx, :])[-n_tau_r:]
            c_proposal = np.copy(c); c_proposal[similar_indices] = len(np.unique(c)) + 1
            unique_clusters_proposal = np.unique(c_proposal)
            for i in range(self.n_subjects):
                log_probs = []
                for k in unique_clusters_proposal:
                    members_k = np.where(c_proposal == k)[0]
                    log_likelihood = self._log_posterior_predictive(X[i], X[members_k, :], self.p, mu0, lambda0, Psi0, nu0)
                    log_probs.append(np.log(np.sum(self.similarity_matrix[i, members_k]) + 1e-9) + log_likelihood)
                probs = np.exp(log_probs - np.max(log_probs)); c[i] = np.random.choice(unique_clusters_proposal, p=probs/np.sum(probs))
            c = pd.factorize(c)[0] + 1
            if t >= self.n_burn: self.posterior_samples.append(c.copy())
        self._summarize_posteriors(); print("TIP Clustering finished."); return self.final_labels
    def _summarize_posteriors(self):
        if not self.posterior_samples: self.final_labels = np.ones(self.n_subjects, dtype=int); return
        self.psm = np.zeros((self.n_subjects, self.n_subjects))
        for c_sample in self.posterior_samples: self.psm += (c_sample[:, None] == c_sample)
        self.psm /= len(self.posterior_samples); np.fill_diagonal(self.psm, 1.0)
        condensed_psm_dist = squareform(1 - self.psm); linked = linkage(condensed_psm_dist, method='average')
        last_merges = linked[-20:, 2]
        if len(last_merges) > 1: jumps = np.diff(last_merges); self.k_est = (20 - np.argmax(jumps)) if len(jumps) > 0 else 2
        else: self.k_est = 1
        print(f"Estimated number of clusters from PSM: {self.k_est}"); self.final_labels = fcluster(linked, self.k_est, criterion='maxclust')

# --- 3. CBTR Implementation (Fully Adaptive Logic) ---
class CBTRClustering:
    def __init__(self, base_time_window=180, angle_cos_threshold=0.1, base_dist_threshold_sq=1e-7, speed_factor=1e-8, turn_factor=1e-9, steady_speed_threshold=0.15):
        self.base_time_window = base_time_window; self.angle_cos_threshold = angle_cos_threshold
        self.base_dist_threshold_sq = base_dist_threshold_sq; self.speed_factor = speed_factor; self.turn_factor = turn_factor
        self.steady_speed_threshold = steady_speed_threshold
    def _predict_pos(self, lat, lon, speed, course, dt):
        knots_to_deg_per_sec = 1.852 / 3600 / 111.32; course_rad = np.deg2rad(course)
        d_lat = speed * knots_to_deg_per_sec * np.cos(course_rad)
        d_lon = speed * knots_to_deg_per_sec * np.sin(course_rad) / np.cos(np.deg2rad(lat))
        return lat + d_lat * dt, lon + d_lon * dt
    def fit(self, df):
        print("Starting CBTR Clustering (Fully Adaptive Logic)...")
        df_sorted = df.sort_values('timestamp'); df_sorted['prev_course'] = df_sorted.groupby('mmsi')['course'].shift(1)
        df_sorted['course_change'] = np.abs(df_sorted['course'] - df_sorted['prev_course'])
        df_sorted['course_change'] = df_sorted['course_change'].apply(lambda x: 360 - x if x > 180 else x); df_sorted['course_change'].fillna(0, inplace=True)
        data = df_sorted.to_dict('records')
        n = len(data); bpnp_links = -np.ones(n, dtype=int); min_distances = np.full(n, np.inf)
        print("Finding Best Possible Next Point (BPNP) for each point...")
        for i in range(n):
            xi = data[i]; dynamic_time_window = max(30, self.base_time_window * (xi['speed'] / 15.0 if xi['speed'] > 1 else 0.2))
            best_j, min_D = -1, np.inf
            for j in range(i + 1, n):
                if data[j]['time_sec'] > xi['time_sec'] + dynamic_time_window: break
                xj = data[j]; dt = xj['time_sec'] - xi['time_sec']; avg_speed = (xi['speed'] + xj['speed']) / 2.0
                if dt <= 0: continue
                if avg_speed < self.steady_speed_threshold: D = (xi['lat'] - xj['lat'])**2 + (xi['lon'] - xj['lon'])**2
                else:
                    gamma_i_lat, gamma_i_lon = self._predict_pos(xi['lat'], xi['lon'], xi['speed'], xi['course'], dt)
                    d_fwd_sq = (gamma_i_lat - xj['lat'])**2 + (gamma_i_lon - xj['lon'])**2
                    sigma_j_lat, sigma_j_lon = self._predict_pos(xj['lat'], xj['lon'], xj['speed'], xj['course'], -dt)
                    d_bwd_sq = (sigma_j_lat - xi['lat'])**2 + (sigma_j_lon - xi['lon'])**2; D = 0.5 * (d_fwd_sq + d_bwd_sq)
                    alpha = 1.0 / np.cos(np.deg2rad(xi['lat'])); tau = 4e-5 if xi['speed'] > 4 else 4e-6
                    vec_ij = np.array([tau * dt, alpha * (xj['lat'] - xi['lat']), xj['lon'] - xi['lon']])
                    vec_pred = np.array([tau * dt, alpha * (gamma_i_lat - xi['lat']), gamma_i_lon - xi['lon']])
                    cos_phi = np.dot(vec_ij, vec_pred) / (np.linalg.norm(vec_ij) * np.linalg.norm(vec_pred) + 1e-9)
                    if cos_phi < self.angle_cos_threshold: continue
                if D < min_D: min_D, best_j = D, j
            if best_j != -1: bpnp_links[i], min_distances[i] = best_j, min_D
        print("Forming clusters using adaptive thresholds and graph components...")
        adj = defaultdict(list)
        for i in range(n):
            j = bpnp_links[i]
            if j != -1:
                xi = data[i]; adaptive_threshold = self.base_dist_threshold_sq * (1 + self.speed_factor * xi['speed']**2) * (1 + self.turn_factor * xi['course_change']**2)
                if min_distances[i] < adaptive_threshold: adj[i].append(int(j)); adj[int(j)].append(i)
        cluster_labels = -np.ones(n, dtype=int); current_cluster_id = 0; visited = set()
        for i in range(n):
            if i not in visited:
                current_cluster_id += 1; component = []; q = [i]; visited.add(i)
                while q:
                    u = q.pop(0); component.append(u)
                    for v in adj[u]:
                        if v not in visited: visited.add(v); q.append(v)
                for node in component: cluster_labels[node] = current_cluster_id
        print("CBTR Clustering finished."); return cluster_labels

# --- 4. Final Hyper-Realistic AIS Simulation ---
TAMPA_LAT, TAMPA_LON = 27.96, -82.45; ST_PETE_LAT, ST_PETE_LON = 27.77, -82.64
BAY_CENTER_LAT = (TAMPA_LAT + ST_PETE_LAT) / 2; BAY_CENTER_LON = (TAMPA_LON + ST_PETE_LON) / 2
LAT_RANGE = abs(TAMPA_LAT - ST_PETE_LAT); LON_RANGE = abs(TAMPA_LON - ST_PETE_LON)
print(f"\n--- Generating Final Hyper-Realistic AIS Data ---")

def generate_trajectory(n_points, behavior, mmsi, reporting_interval):
    start_lat = BAY_CENTER_LAT + np.random.uniform(-LAT_RANGE/3, LAT_RANGE/3)
    start_lon = BAY_CENTER_LON + np.random.uniform(-LON_RANGE/3, LON_RANGE/3)
    time_deltas = np.random.normal(reporting_interval, reporting_interval * 0.2, n_points).clip(min=2.0)
    timestamps = pd.to_datetime(np.cumsum(time_deltas), unit='s', origin='2024-01-01')
    lons, lats, speeds, courses = [start_lon], [start_lat], [], []
    current_speed = 0; current_course = np.random.uniform(0, 360)
    for i in range(n_points - 1):
        dt = (timestamps[i+1] - timestamps[i]).total_seconds()
        if behavior == 'high_speed': current_speed = np.random.uniform(15, 20); current_course += np.random.normal(0, 0.5)
        elif behavior == 'drifting': current_speed = np.random.uniform(0.1, 1.5); current_course = np.random.uniform(0, 360)
        elif behavior == 'varying_speed':
            if np.random.rand() < 0.2: current_speed = np.random.uniform(0.5, 3) if current_speed > 3 else np.random.uniform(5, 10)
            current_course += np.random.normal(0, 5)
        elif behavior == 'random_walk':
            current_speed = np.random.uniform(6, 10); current_course += np.random.normal(0, 25)
        lat, lon = lats[-1], lons[-1]
        new_lat, new_lon = CBTRClustering()._predict_pos(lat, lon, current_speed, current_course, dt)
        lats.append(new_lat); lons.append(new_lon); speeds.append(current_speed); courses.append(current_course % 360)
    speeds.append(speeds[-1]); courses.append(courses[-1])
    return pd.DataFrame({'mmsi': mmsi, 'lon': lons, 'lat': lats, 'timestamp': timestamps, 'speed': speeds, 'course': courses, 'behavior': behavior})

behaviors = {'high_speed': {'interval': 8}, 'drifting': {'interval': 240},
             'varying_speed': {'interval': 90}, 'random_walk': {'interval': 15}}
behavior_keys = list(behaviors.keys()); num_vessels = 8
trajectories = [generate_trajectory(np.random.randint(50, 80), behavior_keys[i % len(behaviors)],
                                     11111*(i+1), behaviors[behavior_keys[i % len(behaviors)]]['interval'])
                for i in range(num_vessels)]
ais_data = pd.concat(trajectories)
ais_data['time_sec'] = (ais_data['timestamp'] - ais_data['timestamp'].min()).dt.total_seconds()
ais_data = ais_data.sort_values('time_sec').reset_index(drop=True)
ground_truth_labels = ais_data['mmsi']; print("Sample of generated AIS data:"); print(ais_data.head())


# --- 5. Run All Clustering Algorithms and Evaluate ---
results = {}; true_k = ais_data['mmsi'].nunique()
label_arrays = {}
X_loc_scaled = StandardScaler().fit_transform(ais_data[['lon', 'lat']].values)
ais_data['course_x'] = np.cos(np.deg2rad(ais_data['course'])); ais_data['course_y'] = np.sin(np.deg2rad(ais_data['course']))
X_rich_scaled = StandardScaler().fit_transform(ais_data[['lon', 'lat', 'speed', 'course_x', 'course_y']].values)

print("\n--- Applying CBTR Clustering (Adaptive Version) ---")
cbtr_model = CBTRClustering(); label_arrays['CBTR (adaptive)'] = cbtr_model.fit(ais_data)
results['CBTR (adaptive)'] = {'ARI': adjusted_rand_score(ground_truth_labels, label_arrays['CBTR (adaptive)']), 'k_found': len(np.unique(label_arrays['CBTR (adaptive)']))}
print("\n--- Applying TIP Clustering (Tuned) ---")
tip_model = TIPClustering(n_burn=200, n_samples=200); label_arrays['TIP (tuned)'] = tip_model.fit(X_rich_scaled)
results['TIP (tuned)'] = {'ARI': adjusted_rand_score(ground_truth_labels, label_arrays['TIP (tuned)']), 'k_found': len(np.unique(label_arrays['TIP (tuned)']))}

# --- NEW: Run GMM (MCLUST) ---
print("\n--- Applying GMM (MCLUST) ---")
gmm = GaussianMixture(n_components=true_k, random_state=42, n_init=10).fit(X_rich_scaled)
label_arrays['GMM (MCLUST)'] = gmm.predict(X_rich_scaled)
results['GMM (MCLUST)'] = {'ARI': adjusted_rand_score(ground_truth_labels, label_arrays['GMM (MCLUST)']), 'k_found': true_k}

print("\n--- Applying k-means Clustering ---")
kmeans = KMeans(n_clusters=true_k, n_init='auto', random_state=42).fit(X_loc_scaled)
label_arrays['k-means'] = kmeans.labels_
results['k-means'] = {'ARI': adjusted_rand_score(ground_truth_labels, label_arrays['k-means']), 'k_found': true_k}
print("\n--- Applying DBSCAN Clustering (Tuned) ---")
dbscan = DBSCAN(eps=0.05, min_samples=5).fit(X_loc_scaled); label_arrays['DBSCAN (tuned)'] = dbscan.labels_
non_noise_mask = dbscan.labels_ != -1
ari_dbscan = adjusted_rand_score(ground_truth_labels[non_noise_mask], dbscan.labels_[non_noise_mask]) if sum(non_noise_mask) > 0 else 0
results['DBSCAN (tuned)'] = {'ARI': ari_dbscan, 'k_found': len(np.unique(dbscan.labels_[non_noise_mask]))}
print("\n--- Applying Hierarchical Clustering Methods ---")
hclust_methods = {'H-Clust (Single)': 'single', 'H-Clust (Complete)': 'complete', 'H-Clust (Average)': 'average', 'H-Clust (Centroid)': 'centroid'}
for name, method in hclust_methods.items():
    print(f"  - Running {name}...")
    linked = linkage(X_loc_scaled, method=method); label_arrays[name] = fcluster(linked, t=true_k, criterion='maxclust')
    results[name] = {'ARI': adjusted_rand_score(ground_truth_labels, label_arrays[name]), 'k_found': true_k}

# --- 6. Report and Compare Overall Accuracy Rates ---
print("\n\n--- Overall Clustering Performance Comparison ---\n")
results_df = pd.DataFrame(results).T; results_df['k_true'] = true_k
results_df = results_df[['k_true', 'k_found', 'ARI']].sort_values(by='ARI', ascending=False); print(results_df)

# --- 7. Per-Vessel-Type Accuracy Analysis ---
print("\n\n--- Per-Vessel-Type Accuracy Analysis (ARI Score) ---\n")
for method, labels in label_arrays.items():
    sanitized_name = method.lower().replace(' ', '-').replace('(', '').replace(')', '')
    ais_data[f'{sanitized_name}_labels'] = labels

per_type_results = {}
for behavior in ais_data['behavior'].unique():
    per_type_results[behavior] = {}
    mask = ais_data['behavior'] == behavior
    true_subset_labels = ais_data.loc[mask, 'mmsi']
    for method, labels in label_arrays.items():
        pred_subset_labels = labels[mask]
        ari_score = 0
        if 'DBSCAN' in method:
            sub_non_noise_mask = pred_subset_labels != -1
            if sum(sub_non_noise_mask) > 1: ari_score = adjusted_rand_score(true_subset_labels[sub_non_noise_mask], pred_subset_labels[sub_non_noise_mask])
        else: ari_score = adjusted_rand_score(true_subset_labels, pred_subset_labels)
        per_type_results[behavior][method] = ari_score

per_type_df = pd.DataFrame(per_type_results).T
column_order = ['CBTR (adaptive)', 'TIP (tuned)', 'GMM (MCLUST)', 'DBSCAN (tuned)', 'H-Clust (Single)', 'H-Clust (Complete)', 'H-Clust (Average)', 'H-Clust (Centroid)', 'k-means']
per_type_df = per_type_df[[col for col in column_order if col in per_type_df.columns]]
print(per_type_df)

# --- 8. Visualize Best and Worst Performers on a Map ---
best_method = results_df.index[0]; worst_method = results_df.index[-1]
print(f"\nVisualizing best performer ({best_method}) and worst performer ({worst_method}) on maps.")

def create_map(df, cluster_col, title):
    map_center = [df['lat'].mean(), df['lon'].mean()]; m = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB positron')
    num_clusters = df[cluster_col].nunique()
    colors = sns.color_palette('husl', n_colors=num_clusters).as_hex() if num_clusters > 0 else []
    cluster_colors = {cid: color for cid, color in zip(sorted(df[cluster_col].unique()), colors)}
    for _, point in df.iterrows():
        cid = point[cluster_col]; color = cluster_colors.get(cid, '#000000')
        folium.CircleMarker([point['lat'], point['lon']], radius=3, color=color, fill=True, fill_color=color,
                            fill_opacity=0.8, popup=f"MMSI: {point['mmsi']}<br>Cluster: {cid}").add_to(m)
    title_html = f'<h3 align="center" style="font-size:16px"><b>{title}</b></h3>'
    m.get_root().html.add_child(folium.Element(title_html)); return m

best_col_name = best_method.lower().replace(' ', '-').replace('(', '').replace(')', '') + '_labels'
worst_col_name = worst_method.lower().replace(' ', '-').replace('(', '').replace(')', '') + '_labels'
print(f"\n--- Displaying Map for Best Method: {best_method} ---")
display(create_map(ais_data, best_col_name, f'Best Performer: {best_method} (ARI: {results[best_method]["ARI"]:.3f})'))
print(f"\n--- Displaying Map for Worst Method: {worst_method} ---")
display(create_map(ais_data, worst_col_name, f'Worst Performer: {worst_method} (ARI: {results[worst_method]["ARI"]:.3f})'))
print("\n--- Demonstration Complete ---")


--- Generating Final Hyper-Realistic AIS Data ---
Sample of generated AIS data:
    mmsi        lon        lat                     timestamp      speed  \
0  55555 -82.499174  27.824711 2024-01-01 00:00:07.971322298  16.032106   
1  11111 -82.585098  27.902562 2024-01-01 00:00:08.758177280  15.994212   
2  88888 -82.498273  27.841845 2024-01-01 00:00:10.592761278   6.449217   
3  55555 -82.499740  27.824994 2024-01-01 00:00:15.731517553  18.325183   
4  44444 -82.541220  27.894925 2024-01-01 00:00:17.962479115   8.774870   

       course     behavior  time_sec  
0  299.517316   high_speed  0.000000  
1    5.534028   high_speed  0.786855  
2  253.190918  random_walk  2.621439  
3  300.302563   high_speed  7.760195  
4  305.643311  random_walk  9.991157  

--- Applying CBTR Clustering (Adaptive Version) ---
Starting CBTR Clustering (Fully Adaptive Logic)...
Finding Best Possible Next Point (BPNP) for each point...


/tmp/ipython-input-4-351215151.py:115: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_sorted['course_change'] = df_sorted['course_change'].apply(lambda x: 360 - x if x > 180 else x); df_sorted['course_change'].fillna(0, inplace=True)


Forming clusters using adaptive thresholds and graph components...
CBTR Clustering finished.

--- Applying TIP Clustering (Tuned) ---
Starting TIP Gibbs Sampler (200 burn-in, 200 samples)...
Iteration 100/400
Iteration 200/400
Iteration 300/400
Iteration 400/400
Estimated number of clusters from PSM: 6
TIP Clustering finished.

--- Applying GMM (MCLUST) ---

--- Applying k-means Clustering ---

--- Applying DBSCAN Clustering (Tuned) ---

--- Applying Hierarchical Clustering Methods ---
  - Running H-Clust (Single)...
  - Running H-Clust (Complete)...
  - Running H-Clust (Average)...
  - Running H-Clust (Centroid)...


--- Overall Clustering Performance Comparison ---

                    k_true  k_found       ARI
GMM (MCLUST)             8      8.0  0.798682
H-Clust (Single)         8      8.0  0.781756
DBSCAN (tuned)           8     15.0  0.686122
k-means                  8      8.0  0.659990
H-Clust (Complete)       8      8.0  0.582446
H-Clust (Centroid)       8      8.0  0.566338
H


--- Displaying Map for Worst Method: CBTR (adaptive) ---



--- Demonstration Complete ---
